# Notebook 06 — Multi-Feature SGD with Feature Scaling (Advertising Dataset)

**Session:** Linear Regression — L1, L2, and Huber Losses (Level 2)  
**Module:** 02 · Calculus and Linear Algebra

## Goal

Extend linear regression to **three input features** (TV, Radio, Newspaper budgets) and
apply mean–range feature scaling before training with stochastic gradient descent (SGD).

**Learning objectives:**
- Load and preprocess a real-world multi-feature CSV dataset.
- Implement mean–range scaling and store normalisation statistics.
- Implement multi-feature `predict` and per-feature gradient functions.
- Run SGD and inspect the loss curve.
- Correctly rescale test inputs at inference time.

In [ ]:
import random

import matplotlib.pyplot as plt
import numpy as np

## 1  Load Data

In [ ]:
def get_column(data: list, index: int) -> list:
    return [row[index] for row in data]


data = np.genfromtxt("../data/advertising.csv", delimiter=",", skip_header=1).tolist()
N = len(data)

tv_data = get_column(data, 0)
radio_data = get_column(data, 1)
newspaper_data = get_column(data, 2)
sales_data = get_column(data, 3)

print(f"N              : {N}")
print(f"TV first 5     : {tv_data[:5]}")
print(f"Radio first 5  : {radio_data[:5]}")
print(f"Sales first 5  : {sales_data[:5]}")

## 2  Feature Scaling

Mean–range scaling centres the data around zero and divides by the value range.  We scale
**all three input features jointly** using a shared mean, max, and min.

In [ ]:
def mean_range_scale(
    *feature_lists: list,
) -> tuple[list[list], tuple[float, float, float]]:
    """Apply mean–range scaling to multiple feature lists jointly.

    Returns:
        scaled: list of scaled feature lists (same order as input).
        stats:  (mean_value, max_value, min_value) over the combined pool.
    """
    combined = [v for col in feature_lists for v in col]
    max_v = max(combined)
    min_v = min(combined)
    mean_v = sum(combined) / len(combined)
    spread = max_v - min_v

    scaled = [[(x - mean_v) / spread for x in col] for col in feature_lists]
    return scaled, (mean_v, max_v, min_v)


(tv_scaled, radio_scaled, newspaper_scaled), (mean_v, max_v, min_v) = mean_range_scale(
    tv_data, radio_data, newspaper_data
)

print(f"Scaling stats  →  mean={mean_v:.3f}, max={max_v:.3f}, min={min_v:.3f}")
print(f"TV scaled first 5  : {[round(v, 4) for v in tv_scaled[:5]]}")

## 3  Model and Loss Functions

In [ ]:
def predict(
    x1: float,
    x2: float,
    x3: float,
    w1: float,
    w2: float,
    w3: float,
    b: float,
) -> float:
    """Multi-feature linear prediction."""
    return w1 * x1 + w2 * x2 + w3 * x3 + b


def compute_gradient_wi(xi: float, y: float, y_hat: float) -> float:
    """MSE gradient for weight i:  2 * xi * (y_hat - y)."""
    return 2 * xi * (y_hat - y)


def compute_gradient_b(y: float, y_hat: float) -> float:
    """MSE gradient for bias:  2 * (y_hat - y)."""
    return 2 * (y_hat - y)


def update_weight_wi(wi: float, dl_dwi: float, lr: float) -> float:
    return wi - lr * dl_dwi


def update_weight_b(b: float, dl_db: float, lr: float) -> float:
    return b - lr * dl_db


def initialise_params(seed: int = 42) -> tuple[float, float, float, float]:
    """Initialise weights from N(0, 1) and bias to 0."""
    random.seed(seed)
    w1 = random.gauss(mu=0.0, sigma=1.0)
    w2 = random.gauss(mu=0.0, sigma=1.0)
    w3 = random.gauss(mu=0.0, sigma=1.0)
    b = 0.0
    return w1, w2, w3, b

## 4  SGD Training Loop

In [ ]:
w1, w2, w3, b = initialise_params()
lr: float = 0.01
epoch_max: int = 50
losses: list[float] = []

for _epoch in range(epoch_max):
    for i in range(N):
        x1 = tv_scaled[i]
        x2 = radio_scaled[i]
        x3 = newspaper_scaled[i]
        y = sales_data[i]

        y_hat = predict(x1, x2, x3, w1, w2, w3, b)
        loss = (y_hat - y) ** 2
        losses.append(loss)

        dl_dw1 = compute_gradient_wi(x1, y, y_hat)
        dl_dw2 = compute_gradient_wi(x2, y, y_hat)
        dl_dw3 = compute_gradient_wi(x3, y, y_hat)
        dl_db = compute_gradient_b(y, y_hat)

        w1 = update_weight_wi(w1, dl_dw1, lr)
        w2 = update_weight_wi(w2, dl_dw2, lr)
        w3 = update_weight_wi(w3, dl_dw3, lr)
        b = update_weight_b(b, dl_db, lr)

print(f"Final  w1={w1:.4f}, w2={w2:.4f}, w3={w3:.4f}, b={b:.4f}")

## 5  Loss Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(losses)
axes[0].set_title("Per-sample MSE loss (all)")
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("MSE")

axes[1].plot(losses[-100:])
axes[1].set_title("Per-sample MSE loss (last 100 iterations)")
axes[1].set_xlabel("Iteration")

plt.tight_layout()
plt.show()

## 6  Inference

Scale test inputs using the **same statistics** computed from the training data.

In [ ]:
def scale_test_point(
    tv: float,
    radio: float,
    newspaper: float,
    mean_v: float,
    max_v: float,
    min_v: float,
) -> tuple[float, float, float]:
    spread = max_v - min_v
    return (
        (tv - mean_v) / spread,
        (radio - mean_v) / spread,
        (newspaper - mean_v) / spread,
    )


# Example 1: high-budget campaign
tv_test, radio_test, newspaper_test = scale_test_point(230.1, 37.8, 69.2, mean_v, max_v, min_v)
sales_pred = predict(tv_test, radio_test, newspaper_test, w1, w2, w3, b)
print(f"Predicted sales (230.1, 37.8, 69.2): {sales_pred:.2f}")

# Example 2: low-budget campaign
tv_test, radio_test, newspaper_test = scale_test_point(19.2, 35.9, 51.3, mean_v, max_v, min_v)
sales_pred = predict(tv_test, radio_test, newspaper_test, w1, w2, w3, b)
print(f"Predicted sales (19.2, 35.9, 51.3)  : {sales_pred:.2f}")